In [2]:
import os
from dotenv import load_dotenv
from unstructured.partition.pdf import partition_pdf

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
GROQ_API_KEY = os.getenv("GROQ_API_KEY")
LANGCHAIN_API_KEY = os.getenv("LANGSMITH_API_KEY")
LANGCHAIN_TRACING_V2 = "true"


output_path = "../output/"
file_path = "../content/CIP_EINSTEIN_264.pdf"

/home/bzzmn/projects/domiapp/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
from unstructured.partition.pdf import partition_pdf

# Reference: https://docs.unstructured.io/open-source/core-functionality/chunking
chunks = partition_pdf(
    filename=file_path,
    infer_table_structure=True,            # extract tables
    strategy="hi_res",                     # mandatory to infer tables

    extract_image_block_types=["Image"],   # Add 'Table' to list to extract image of tables
    # image_output_dir_path=output_path,   # if None, images and tables will saved in base64

    extract_image_block_to_payload=True,   # if true, will extract base64 for API usage

    chunking_strategy="by_title",          # or 'basic'
    max_characters=10000,                  # defaults to 500
    combine_text_under_n_chars=2000,       # defaults to 0
    new_after_n_chars=6000,

    # extract_images_in_pdf=True,          # deprecated
)

CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox


In [82]:
len(chunks)

3

In [77]:
chunks

In [78]:
chunks[0].to_dict()

{'type': 'CompositeElement',
 'element_id': '537413765b89f8edcf9a3cc5df92fc98',
 'text': 'El presente documento ha sido suscrito por medio de Firma Electronica Avanzada\n\nhttps://www.sistemasrecoleta.cl/validarDocDigital.php\n\nValidar en Codigo: ff35c0abc1e163f\n\nGSPublisherVersion 0.82.100.100\n\nmunicipalidad Recoleta Somos todos\n\nCERTIFICADO DE INFORMACIONES PREVIAS DIRECCION DE OBRAS - |. MUNICIPALIDAD DE RECOLETA REGION METROPOLITANA\n\nx}\n\nURBANO\n\nRURAL\n\n25.07.2023\n\n1. IDENTIFICACION DE LA PROPIEDAD (certiFicabo DE NUMERO)\n\nAV. EINSTEIN VISTA HERMOSA 3173 - 028 2. INSTRUMENTO(S) DE PLANIFICACION TERRITORIAL APLICABLE(S) P.R.M.S. y sus modificaciones vigentes Res. N° 104 de fecha 22.11.2004 Res N° 2591 Exento de fecha 27.06.2012 Dec N°555 Exento de fecha 12.03.2018 EXTENSION URBANA 11-02-2010 08-01-2005 10-08-2012 16-03-2018 URBANA 3. DECLARATORIA DE POSTERGACION DE PERMISO (Art. 117 LGUC) x Si No 5 NORMAS URBANISTICAS (€n caso necesario se adjunta hoja anexa) 5.1 U

In [80]:
for i, doc in enumerate(chunks):
    if "CompositeElement" in str(type(doc)):
        print("\n\nChunk", i)
        for doc in doc.metadata.orig_elements:
            print(doc.to_dict()["type"], doc.metadata.page_number)
        break







Chunk 0
Title 1
Title 1
Header 1
Image 1
Title 1
Image 1
NarrativeText 1
Title 1
Title 1
Title 1
UncategorizedText 1
FigureCaption 1
Table 1
Title 1
ListItem 1
NarrativeText 1
Title 1
ListItem 1
Table 1
Image 1
Header 1


In [4]:
chunks[0].metadata.orig_elements

In [93]:
elements = chunks[0].metadata.orig_elements
for element in elements:
    if "Table" in str(type(element)):
        print(element.metadata.text_as_html)



<table><thead><tr><th>LOTEO</th><th colspan="2">VISTA HERMOSA</th><th>MANZANA</th><th>22</th><th>LOTE</th><th>66</th></tr></thead><tbody><tr><td>mows</td><td>3173 - 028</td><td>ASIGNADO</td><td>N°</td><td></td><td>N° 264</td><td></td></tr><tr><td colspan="7">EL 2. INSTRUMENTO(S) DE PLANIFICACION TERRITORIAL APLICABLE(S)</td></tr><tr><td>PLAN REGULADOR</td><td>INTERCOMUNAL O METROPOLITANO</td><td></td><td>P.R.M.S. y sus modificaciones</td><td>vigentes</td><td>Fecha Diario Oficial</td><td>11-02-2010</td></tr><tr><td>PLAN REGULADOR</td><td>COMUNAL</td><td></td><td>Res. N2 104 de fecha</td><td>22.11.2004</td><td>Fecha Diario Oficial</td><td>08-01-2005</td></tr><tr><td>MODIFICACION N?</td><td>1 PLAN REGULADOR COMUNAL</td><td></td><td>Res N2 2591 Exento de</td><td>fecha 27.06.2012</td><td>Fecha Diario Oficial</td><td>10-08-2012</td></tr><tr><td>MODIFICACION N°</td><td>2 PLAN REGULADOR COMUNAL</td><td></td><td>Dec N°555 Exento de</td><td>fecha 12.03.2018</td><td>Fecha Diario Oficial</td><td>1

In [ ]:
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_openai import ChatOpenAI
from base64 import b64decode
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

docs = chunks



def parse_docs(docs):
    context_text = ""
    for doc in docs:
        for element in doc.metadata.orig_elements:
            if "Table" in str(type(element)):
                context_text += element.metadata.text_as_html
            else:
                context_text += element.text
    return context_text

# Definir el template
template = """Responde la pregunta basada unicamente en el siguiente contexto, que puede incluir texto y tablas.
Contexto: {context}
Pregunta: {question}"""

prompt = ChatPromptTemplate.from_template(template)
model = ChatGroq(temperature=0.5, model="gemma2-9b-it")
chain = prompt | model | StrOutputParser()

# Preparar los datos
context = parse_docs(docs)


prompt_data = {
    "context": context,  # Asegúrate de formatear el contexto apropiadamente
    "question": ""
}

# Hacer la llamada
response = chain.invoke(prompt_data)
print(response)

No se encuentra el número de certificado en el texto proporcionado. 



